#### Error Mitigated QSVM (ZNE + REM Combined) - Spambase

This notebook implements **combined error mitigation** using:
- **Zero-Noise Extrapolation (ZNE)**: Extrapolates results from multiple noise scales
- **Readout Error Mitigation (REM)**: Corrects measurement errors using calibration matrix

Focus: Comparing generalization capability of Error-Mitigated QSVM vs Classical SVM

In [ ]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

In [ ]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [ ]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score

In [ ]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

##### Load Dataset

In [ ]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d", "word_freq_our",
    "word_freq_over", "word_freq_remove", "word_freq_internet", "word_freq_order", "word_freq_mail",
    "word_freq_receive", "word_freq_will", "word_freq_people", "word_freq_report", "word_freq_addresses",
    "word_freq_free", "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money", "word_freq_hp",
    "word_freq_hpl", "word_freq_george", "word_freq_650", "word_freq_lab", "word_freq_labs",
    "word_freq_telnet", "word_freq_857", "word_freq_data", "word_freq_415", "word_freq_85",
    "word_freq_technology", "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project", "word_freq_re",
    "word_freq_edu", "word_freq_table", "word_freq_conference", "char_freq_;", "char_freq_(",
    "char_freq_[", "char_freq_!", "char_freq_$", "char_freq_#", "capital_run_length_average",
    "capital_run_length_longest", "capital_run_length_total", "label"
]

# --- Load the Spambase Dataset ---
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")

##### Noise Model and Error Mitigation Functions

In [ ]:
# Base error rates (realistic NISQ device)
NOISE_CONFIGS = {
    'low': {'p_1q': 0.0001, 'p_2q': 0.001, 'p_ro': 0.002},
    'standard': {'p_1q': 0.001,  'p_2q': 0.01,  'p_ro': 0.02},
    'high': {'p_1q': 0.005,  'p_2q': 0.05,  'p_ro': 0.10}
}

def get_scaled_noise_model(scale_factor=1.0, level='standard'):
    """
    Build a noise model with scaled error probabilities for ZNE.
    Uses formula: p_scaled = 1 - (1-p)^scale_factor
    """
    config = NOISE_CONFIGS.get(level, NOISE_CONFIGS['standard'])
    
    p_1q = config['p_1q']
    p_2q = config['p_2q']
    p_ro = config['p_ro']

    p_1q_scaled = 1 - (1 - p_1q)**scale_factor
    p_2q_scaled = 1 - (1 - p_2q)**scale_factor
    p_ro_scaled = 1 - (1 - p_ro)**scale_factor
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_1q_scaled, 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_2q_scaled, 2), ['cx'])
    readout_error = ReadoutError([[1 - p_ro_scaled, p_ro_scaled], [p_ro_scaled, 1 - p_ro_scaled]])
    noise_model.add_all_qubit_readout_error(readout_error)
    
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    
    return noise_model, backend, pm, config

print("Noise model factory function ready!")

In [ ]:
# ==========================================
# READOUT ERROR MITIGATION (REM) FUNCTION
# ==========================================

def apply_rem_to_kernel(kernel_matrix, p_ro, n_qubits):
    """
    Apply Readout Error Mitigation to a kernel matrix.
    
    The correction formula for fidelity with symmetric readout error:
    K_corrected = (K_noisy - bias) / correction_factor
    """
    correction_factor = (1 - 2 * p_ro) ** n_qubits
    
    if abs(correction_factor) < 1e-10:
        print("Warning: Correction factor too small, using raw kernel")
        return kernel_matrix
    
    bias = 0.5 * (1 - correction_factor)
    corrected_kernel = (kernel_matrix - bias) / correction_factor
    
    # Clip to valid kernel range [0, 1]
    corrected_kernel = np.clip(corrected_kernel, 0, 1)
    
    # Ensure diagonal is exactly 1 (self-similarity)
    if kernel_matrix.shape[0] == kernel_matrix.shape[1]:
        np.fill_diagonal(corrected_kernel, 1.0)
    
    return corrected_kernel

print("REM function ready!")

##### Experiment Configurations (ZNE+REM Combined Only)

In [ ]:
# ==========================================
# EXPERIMENT CONFIGURATIONS (ZNE+REM COMBINED)
# ==========================================
#
# All experiments use combined ZNE + REM error mitigation
# for best-case error-mitigated QSVM performance.
#
# ==========================================

experiments = [
    # --- EXP 1: Sample Size Effect ---
    {'id': 'Exp1_100samp',  'samples': 100, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_200samp',  'samples': 200, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_300samp',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_400samp',  'samples': 400, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_500samp',  'samples': 500, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 2: Dimensionality Effect ---
    {'id': 'Exp2_2feat',   'samples': 300, 'k_features': 2,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_4feat',   'samples': 300, 'k_features': 4,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_6feat',   'samples': 300, 'k_features': 6,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_8feat',   'samples': 300, 'k_features': 8,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_10feat',  'samples': 300, 'k_features': 10, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    # {'id': 'Exp2_12feat',  'samples': 300, 'k_features': 12, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 3: Shot Noise Effect ---
    {'id': 'Exp3_128shots',  'samples': 300, 'k_features': 8, 'shots': 128,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_512shots',  'samples': 300, 'k_features': 8, 'shots': 512,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_1024shots', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 4: Reps Effect ---
    {'id': 'Exp4_Reps1', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp4_Reps2', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp4_Reps3', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 3, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 5: Entanglement Effect ---
    {'id': 'Exp5_Linear',   'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear',   'noise_level': 'standard'},
    {'id': 'Exp5_Circular', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'circular', 'noise_level': 'standard'},
    {'id': 'Exp5_Full',     'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'full',     'noise_level': 'standard'},

    # --- EXP 6: Noise Level Effect ---
    {'id': 'Exp6_LowNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low'},
    {'id': 'Exp6_StdNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp6_HighNoise', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high'},
]

print(f"Total experiments configured: {len(experiments)}")
print("All experiments use combined ZNE+REM error mitigation")

##### Main Experiment Loop (ZNE+REM Combined)

In [ ]:
from qiskit_machine_learning.utils import algorithm_globals

# ZNE scales for Richardson extrapolation
ZNE_SCALES = [1.0, 3.0]

all_results = []

for i, config in enumerate(experiments, 1):
    print("="*80)
    print(f"EXPERIMENT {i}/{len(experiments)}: {config['id']} (ZNE+REM Mitigated)")
    print("="*80)
    print(f"  Samples: {config['samples']}")
    print(f"  K Features: {config['k_features']}")
    print(f"  Noise Level: {config['noise_level']}")
    
    start_time = time.time()
    
    # --- 1. Data Preparation ---
    X = df.drop('label', axis=1)
    y = df['label']
    
    X_subset, _, y_subset, _ = train_test_split(
        X, y,
        train_size=config['samples'],
        stratify=y,
        random_state=42
    )
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        test_size=0.30,
        random_state=42,
        stratify=y_subset
    )
    
    # Scaling & Feature Selection
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)
    
    # Drop highly correlated features
    corr_matrix = X_train_scaled.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
    X_train_scaled.drop(columns=to_drop, inplace=True)
    X_test_scaled.drop(columns=to_drop, inplace=True)
    
    # SelectKBest
    k = config['k_features']
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
    X_test_kbest = selector.transform(X_test_scaled)
    
    # Get noise config for REM
    noise_config = NOISE_CONFIGS.get(config['noise_level'], NOISE_CONFIGS['standard'])
    p_ro = noise_config['p_ro']
    n_qubits = k
    
    # --- 2. ZNE: Compute Kernels for Each Scale ---
    kernels_train = {}
    kernels_test = {}
    
    feature_map = ZZFeatureMap(
        feature_dimension=k, 
        reps=config['reps'], 
        entanglement=config['entanglement']
    )
    
    for scale in ZNE_SCALES:
        print(f"  Computing kernel for scale={scale}...")
        _, backend, pm, _ = get_scaled_noise_model(scale_factor=scale, level=config['noise_level'])
        sampler = AerSampler.from_backend(backend=backend, default_shots=config['shots'])
        fidelity = ComputeUncompute(sampler=sampler, pass_manager=pm)
        qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
        
        kernels_train[scale] = qkernel.evaluate(x_vec=X_train_kbest)
        kernels_test[scale] = qkernel.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
    
    # --- 3. Apply ZNE (Richardson Extrapolation) ---
    # Linear 2-point: K_zne = 1.5 * K(scale=1) - 0.5 * K(scale=3)
    kernel_train_zne = 1.5 * kernels_train[1.0] - 0.5 * kernels_train[3.0]
    kernel_test_zne = 1.5 * kernels_test[1.0] - 0.5 * kernels_test[3.0]
    print("  ZNE extrapolation applied.")
    
    # --- 4. Apply REM on ZNE result ---
    kernel_train_final = apply_rem_to_kernel(kernel_train_zne, p_ro, n_qubits)
    kernel_test_final = apply_rem_to_kernel(kernel_test_zne, p_ro, n_qubits)
    print(f"  REM correction applied (p_ro={p_ro}, n_qubits={n_qubits}).")
    
    # Ensure kernel values are valid
    kernel_train_final = np.clip(kernel_train_final, 0, 1)
    kernel_test_final = np.clip(kernel_test_final, 0, 1)
    
    # --- 5. Train SVC ---
    param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    svc = SVC(kernel='precomputed', class_weight='balanced')
    grid = GridSearchCV(svc, param_grid, cv=cv, scoring='accuracy')
    grid.fit(kernel_train_final, y_train)
    best_model = grid.best_estimator_
    
    # --- 6. Evaluation ---
    y_train_pred = best_model.predict(kernel_train_final)
    y_test_pred = best_model.predict(kernel_test_final)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred, pos_label=1)
    gen_gap = abs(train_acc - test_acc)
    
    elapsed_time = time.time() - start_time
    
    print(f"  → Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    print(f"  → Spam Recall: {recall:.4f} | Gen Gap: {gen_gap:.4f}")
    print(f"  → Best C: {grid.best_params_['C']} | Time: {elapsed_time:.1f}s")
    
    all_results.append({
        'experiment_id': config['id'],
        'samples': config['samples'],
        'k_features': config['k_features'],
        'shots': config['shots'],
        'reps': config['reps'],
        'entanglement': config['entanglement'],
        'noise_level': config['noise_level'],
        'train_acc': train_acc,
        'test_acc': test_acc,
        'spam_recall': recall,
        'gen_gap': gen_gap,
        'best_c': grid.best_params_['C'],
        'cv_score': grid.best_score_,
        'time_seconds': elapsed_time
    })

# Save Results
results_df = pd.DataFrame(all_results)
results_df.to_csv('em_qsvm_spambase_znerem_results.csv', index=False)
print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETE!")
print("Results saved to: em_qsvm_spambase_znerem_results.csv")
print("="*80)

##### Results Summary

In [ ]:
# Display summary table
print("\nResults Summary (ZNE+REM Error Mitigated QSVM):")
print(results_df[['experiment_id', 'samples', 'k_features', 'noise_level', 'test_acc', 'spam_recall', 'gen_gap']].to_string(index=False))

In [ ]:
# ==========================================
# FIND BEST CONFIGURATIONS
# ==========================================

print("\n" + "=" * 80)
print("BEST CONFIGURATIONS")
print("=" * 80)

# Best overall test accuracy
best_acc_idx = results_df['test_acc'].idxmax()
best_acc_config = results_df.iloc[best_acc_idx]

print("\n BEST TEST ACCURACY:")
print(f"  Experiment: {best_acc_config['experiment_id']}")
print(f"  Test Accuracy: {best_acc_config['test_acc']:.4f}")
print(f"  Spam Recall: {best_acc_config['spam_recall']:.4f}")
print(f"  Gen Gap: {best_acc_config['gen_gap']:.4f}")

# Best spam recall
best_recall_idx = results_df['spam_recall'].idxmax()
best_recall_config = results_df.iloc[best_recall_idx]

print("\n BEST SPAM RECALL:")
print(f"  Experiment: {best_recall_config['experiment_id']}")
print(f"  Test Accuracy: {best_recall_config['test_acc']:.4f}")
print(f"  Spam Recall: {best_recall_config['spam_recall']:.4f}")
print(f"  Gen Gap: {best_recall_config['gen_gap']:.4f}")

# Best generalization (lowest gap)
best_gen_idx = results_df['gen_gap'].idxmin()
best_gen_config = results_df.iloc[best_gen_idx]

print("\n BEST GENERALIZATION (Lowest Gap):")
print(f"  Experiment: {best_gen_config['experiment_id']}")
print(f"  Test Accuracy: {best_gen_config['test_acc']:.4f}")
print(f"  Spam Recall: {best_gen_config['spam_recall']:.4f}")
print(f"  Gen Gap: {best_gen_config['gen_gap']:.4f}")

##### Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (18, 5)

def plot_experiment_group(df, group_prefix, param_col, xlabel, log_x=False):
    """Plot results for a specific experiment group."""
    subset = df[df['experiment_id'].str.contains(group_prefix)].copy()
    if subset.empty:
        print(f"No data for {group_prefix}")
        return
    
    subset = subset.sort_values(param_col)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Performance
    axes[0].plot(subset[param_col], subset['test_acc'], 'o-', label='Test Accuracy', color='#1f77b4')
    axes[0].plot(subset[param_col], subset['spam_recall'], 's--', label='Spam Recall', color='#ff7f0e')
    if log_x: axes[0].set_xscale('log', base=2)
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel('Score')
    axes[0].set_title(f'Performance vs {xlabel}')
    axes[0].set_ylim(0, 1.05)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Generalization Gap
    axes[1].plot(subset[param_col], subset['gen_gap'], 'D-', color='#d62728')
    if log_x: axes[1].set_xscale('log', base=2)
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel('Gap')
    axes[1].set_title('Generalization Gap (Lower is Better)')
    axes[1].grid(True, alpha=0.3)
    
    # Time
    axes[2].plot(subset[param_col], subset['time_seconds'], '^-', color='#2ca02c')
    if log_x: axes[2].set_xscale('log', base=2)
    axes[2].set_xlabel(xlabel)
    axes[2].set_ylabel('Time (s)')
    axes[2].set_title('Computational Cost')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'{group_prefix} Experiments (ZNE+REM)', fontsize=14)
    plt.tight_layout()
    plt.show()

# Plot each experiment group
plot_experiment_group(results_df, 'Exp1', 'samples', 'Training Samples')
plot_experiment_group(results_df, 'Exp2', 'k_features', 'Feature Dimension (Qubits)')
plot_experiment_group(results_df, 'Exp3', 'shots', 'Shots', log_x=True)

In [ ]:
# ==========================================
# HEATMAP: All Experiments Overview
# ==========================================

plt.figure(figsize=(12, 8))

heatmap_data = results_df.set_index('experiment_id')[['test_acc', 'spam_recall', 'gen_gap', 'train_acc', 'cv_score']]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            center=0.75, linewidths=.5, cbar_kws={'label': 'Score'})

plt.title('ZNE+REM Error Mitigated QSVM - Spambase', fontsize=14, pad=20)
plt.ylabel('Experiment ID')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('em_qsvm_spambase_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()